# Cohere Multilingual Embedding → Titan統合Ensemble → Submission

Amazon Bedrockの`cohere.embed-multilingual-v3`で日本語テキストを1024次元へ変換し、E1〜E6 / T1〜T3を評価します。その後、Notebook 04で保存したTitan OOFも結合し、ROC-AUC hill climbingと複数submission作成までを一続きで実行します。

Cohere Embed v3の分類用途に合わせて`input_type='classification'`を使います。各入力は最大512 token（約2,048文字）なのでローカルでも2,048文字に制限し、Bedrock側にも`truncate='END'`を指定します。APIを呼ぶセルはすべて初期状態で無効です。

In [ ]:
from pathlib import Path
import logging
import os
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from embedding_features import (
    build_bedrock_cohere_classification_request,
    embed_bedrock_cohere,
    generate_embeddings,
    load_embeddings,
    parse_bedrock_cohere_response,
)
from ensemble import (
    combine_embedding_oof_sources,
    evaluate_fold_auc,
    hill_climb_auc,
    make_profile_submissions,
    save_ensemble_outputs,
    split_prediction_namespace,
)
from modeling import (
    default_modeling_config,
    fit_full_and_predict_test,
    run_all_experiments,
)
from validation import encode_binary_target, make_time_series_cv

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

## 1. データ列とBedrock Cohere設定

AWS上ではexecution role、ローカルではAWS profileなど、boto3の標準credential chainを使います。Access keyをNotebookへ直接書かないでください。実行roleには対象modelへの`bedrock:InvokeModel`権限が必要です。`NUMERIC_COLS`と`CATEGORICAL_COLS`は確定データに合わせて編集します。

In [ ]:
TARGET_COL = 'science_tech_decision'
ID_COL = 'project_id'
YEAR_COL = 'project_start_year'
PROJECT_COL = 'project_name'
TEXT_COLS = ['project_name', 'project_objective', 'project_summary', 'current_issues']
NUMERIC_COLS = ['project_start_year', 'project_end_year', 'project_fiscal_year', 'budget']
CATEGORICAL_COLS = ['responsible_ministry']

MODEL_ID = 'cohere.embed-multilingual-v3'
REGION_NAME = os.getenv('AWS_REGION') or os.getenv('AWS_DEFAULT_REGION') or 'ap-northeast-1'
AWS_PROFILE_NAME = os.getenv('AWS_PROFILE')  # AWS上では通常None
EMBEDDING_DIM = 1024
ADAPTER_ID = 'cohere-embed-v3-classification-end-batch-v1'
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'embeddings'
BATCH_SIZE = 32  # Cohereは最大96 texts/call。まず32で安全に開始する。
MAX_INPUT_TOKENS = 512
CHARACTERS_PER_TOKEN_ESTIMATE = 4.0  # 512 token ≈ 2,048文字
PRICE_PER_MILLION_TOKENS = 0.0  # 実行直前にmodel/regionの現行入力料金を設定
MAX_BUDGET_USD = 20.0

RUN_SMOKE_API = False
RUN_EMBEDDING_API = False
EXISTING_COHERE_CACHE_DIR = None  # 再開時は生成済みcache directoryをPathで指定
RUN_CV = False
RUN_ENSEMBLE = False
RUN_FINAL_SUBMISSION = False
INCLUDE_TITAN_IN_ENSEMBLE = True
TITAN_OOF_PATH = PROJECT_ROOT / 'outputs' / 'oof_predictions.parquet'  # Notebook 04のOOF
TITAN_EMBEDDING_CACHE_DIR = None  # 最終予測時にNotebook 03のTitan cacheを指定

train = pd.read_csv(PROJECT_ROOT / 'input' / 'train.csv')
test = pd.read_csv(PROJECT_ROOT / 'input' / 'test.csv')
train[TARGET_COL] = encode_binary_target(train[TARGET_COL])  # 該当=1, 非該当=0
print('train:', train.shape, 'test:', test.shape)

## 2. dry-run（API呼び出しなし）

行数、truncate候補、推定token数と設定料金に基づく概算を確認します。`PRICE_PER_MILLION_TOKENS=0`のまま有料セルを有効化すると安全のため停止します。料金は実行時点のAmazon Bedrock Pricingで確認してください。

In [ ]:
embedding_kwargs = dict(
    provider='bedrock',
    model=MODEL_ID,
    output_root=OUTPUT_ROOT,
    text_cols=TEXT_COLS,
    project_id_col=ID_COL,
    target_col=TARGET_COL,
    embedding_dim=EMBEDDING_DIM,
    batch_size=BATCH_SIZE,
    max_input_tokens=MAX_INPUT_TOKENS,
    gemini_chars_per_token=CHARACTERS_PER_TOKEN_ESTIMATE,
    price_per_million_tokens=PRICE_PER_MILLION_TOKENS,
    max_budget_usd=MAX_BUDGET_USD,
    region_name=REGION_NAME,
    aws_profile_name=AWS_PROFILE_NAME,
    request_builder=build_bedrock_cohere_classification_request,
    response_parser=parse_bedrock_cohere_response,
    embedder=embed_bedrock_cohere,
    adapter_id=ADAPTER_ID,
)

train_dry = generate_embeddings(train, split='train', dry_run=True, **embedding_kwargs)
test_dry = generate_embeddings(test, split='test', dry_run=True, **embedding_kwargs)
dry_run_report = pd.DataFrame([train_dry['report'], test_dry['report']])
display(dry_run_report)
print('combined estimated tokens:', int(dry_run_report['total_estimated_tokens'].sum()))
TOTAL_ESTIMATED_COST_USD = float(dry_run_report['estimated_cost_usd'].sum())
print('combined estimated USD:', TOTAL_ESTIMATED_COST_USD)
if TOTAL_ESTIMATED_COST_USD > MAX_BUDGET_USD:
    print('WARNING: train + testの概算がMAX_BUDGET_USDを超えています。')

## 3. 5行smoke test → train/test全件生成

最初に`RUN_SMOKE_API=True`だけでmodel access、region、IAM、response shapeを確認します。成功後に`RUN_EMBEDDING_API=True`で全件生成します。batchごとにshardが保存されるため、中断後も同じ設定なら続きから再開します。

In [ ]:
if RUN_SMOKE_API:
    if PRICE_PER_MILLION_TOKENS <= 0:
        raise RuntimeError('現行のPRICE_PER_MILLION_TOKENSを設定してください。')
    smoke_result = generate_embeddings(
        train.head(5),
        split='train_smoke',
        output_root=PROJECT_ROOT / 'data' / 'embeddings_smoke_cohere',
        dry_run=False,
        **{key: value for key, value in embedding_kwargs.items() if key != 'output_root'},
    )
    assert smoke_result['embeddings'].shape == (min(5, len(train)), EMBEDDING_DIM)
    assert np.isfinite(smoke_result['embeddings']).all()
    print('smoke test passed:', smoke_result['cache_dir'])
else:
    print('Smoke API is disabled. Set RUN_SMOKE_API=True after checking the dry-run.')

In [ ]:
train_embeddings = test_embeddings = None
train_embedding_metadata = test_embedding_metadata = None
cohere_cache_dir = None

if RUN_EMBEDDING_API:
    if PRICE_PER_MILLION_TOKENS <= 0:
        raise RuntimeError('現行のPRICE_PER_MILLION_TOKENSを設定してください。')
    if TOTAL_ESTIMATED_COST_USD > MAX_BUDGET_USD:
        raise RuntimeError('train + testの概算がMAX_BUDGET_USDを超えています。')
    train_result = generate_embeddings(train, split='train', dry_run=False, **embedding_kwargs)
    test_result = generate_embeddings(test, split='test', dry_run=False, **embedding_kwargs)
    if train_result['cache_dir'] != test_result['cache_dir']:
        raise RuntimeError('train/testのCohere cache設定が一致しません。')
    cohere_cache_dir = train_result['cache_dir']
    train_embeddings, train_embedding_metadata = train_result['embeddings'], train_result['metadata']
    test_embeddings, test_embedding_metadata = test_result['embeddings'], test_result['metadata']
elif EXISTING_COHERE_CACHE_DIR is not None:
    cohere_cache_dir = Path(EXISTING_COHERE_CACHE_DIR)
    train_embeddings, train_embedding_metadata = load_embeddings(
        cohere_cache_dir, 'train', expected_df=train, project_id_col=ID_COL
    )
    test_embeddings, test_embedding_metadata = load_embeddings(
        cohere_cache_dir, 'test', expected_df=test, project_id_col=ID_COL
    )

if train_embeddings is not None:
    assert train_embeddings.shape == (len(train), EMBEDDING_DIM)
    assert test_embeddings.shape == (len(test), EMBEDDING_DIM)
    print('Cohere cache:', cohere_cache_dir)
    print('train/test embeddings:', train_embeddings.shape, test_embeddings.shape)
else:
    print('Embedding未読込です。API生成またはEXISTING_COHERE_CACHE_DIRを指定してください。')

## 4. 時系列CVでE1〜E6 / T1〜T3を実験

最新の実在3年度をvalidationとする同じexpanding-window foldで比較します。Cohere Embeddingを使う実験に加え、tabularとTF-IDFのbaselineも同じOOFへ保存します。T4 x1向け設定です。

In [ ]:
folds, cv_diagnostics = make_time_series_cv(
    train,
    year_col=YEAR_COL,
    project_col=PROJECT_COL,
    target_col=TARGET_COL,
    n_valid_years=3,
)
display(cv_diagnostics)

COHERE_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'cohere'
CONFIG = default_modeling_config()
CONFIG['output_dir'] = str(COHERE_OUTPUT_DIR)
CONFIG['tfidf_feature_output_dir'] = str(PROJECT_ROOT / 'data' / 'csv' / 'cohere_tfidf_shared')
CONFIG['text_cols'] = TEXT_COLS
CONFIG['metric'] = 'roc_auc'
CONFIG['e6_pca_dims'] = [None]
CONFIG['mlp']['device'] = 'cuda'
CONFIG['mlp']['use_amp'] = True
CONFIG['mlp']['early_stop_metric'] = 'auc'
CONFIG['catboost']['task_type'] = 'GPU'
CONFIG['catboost']['devices'] = '0'
CONFIG['catboost']['eval_metric'] = 'AUC'
CONFIG['xgboost']['device'] = 'cuda'
CONFIG['xgboost']['tree_method'] = 'hist'
CONFIG['xgboost']['eval_metric'] = 'auc'
CONFIG['run_final_test_prediction'] = False

suite = None
if RUN_CV:
    if train_embeddings is None:
        raise RuntimeError('先にCohere train Embeddingを生成または読込してください。')
    suite = run_all_experiments(
        train=train, folds=folds,
        train_embeddings=train_embeddings,
        train_embedding_metadata=train_embedding_metadata,
        numeric_cols=NUMERIC_COLS, categorical_cols=CATEGORICAL_COLS,
        config=CONFIG, target_col=TARGET_COL, project_col=PROJECT_COL,
        project_id_col=ID_COL, year_col=YEAR_COL,
    )
    display(suite.summary.sort_values('mean', ascending=False))
    display(suite.fold_metrics)
else:
    print('CV is disabled. Set RUN_CV=True after loading Cohere embeddings.')

## 5. Cohere + Titanを複数の年度重みでROC-AUC hill climbing

Cohere OOFのEmbedding依存モデルを`cohere__...`、Notebook 04のTitan OOFを`titan__...`として結合します。Embeddingを使わないE5/T1/T2はCohere側の結果を一度だけ残し、重複候補にしません。fold weightはtest予測を直接年度別に混ぜる値ではなく、hill climbingがモデルweightを選ぶ目的関数です。

In [ ]:
SELECTED_ENSEMBLE_PROFILE = 'recent_20_30_50_rank'
ENSEMBLE_CANDIDATES = None
EMBEDDING_FREE_MODELS = {'E5_tabular_catboost', 'T1_tfidf_lr', 'T2_tfidf_tabular_lr'}
ENSEMBLE_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'cohere_titan_ensemble'
ENSEMBLE_PROFILES = {
    'uniform_rank': {'objective': 'weighted_fold_auc', 'blend_mode': 'rank', 'fold_weights': [1, 1, 1]},
    'recent_20_30_50_rank': {'objective': 'weighted_fold_auc', 'blend_mode': 'rank', 'fold_weights': [0.2, 0.3, 0.5]},
    'recent_10_20_70_rank': {'objective': 'weighted_fold_auc', 'blend_mode': 'rank', 'fold_weights': [0.1, 0.2, 0.7]},
    'recent_20_30_50_probability': {'objective': 'weighted_fold_auc', 'blend_mode': 'probability', 'fold_weights': [0.2, 0.3, 0.5]},
}
ensemble_results = {}
ensemble_result = None
combined_oof_predictions = None

if RUN_ENSEMBLE:
    if suite is None:
        raise RuntimeError('先にRUN_CV=TrueでCohere OOFを作成してください。')
    prediction_sources = {'cohere': suite.oof_predictions}
    if INCLUDE_TITAN_IN_ENSEMBLE:
        if not TITAN_OOF_PATH.exists():
            raise FileNotFoundError(
                f'Titan OOFがありません: {TITAN_OOF_PATH}。先にNotebook 04でRUN_CV=Trueを実行してください。'
            )
        prediction_sources['titan'] = pd.read_parquet(TITAN_OOF_PATH)
    combined_oof_predictions = combine_embedding_oof_sources(
        prediction_sources,
        embedding_free_models=EMBEDDING_FREE_MODELS,
        shared_source='cohere',
    )
    if INCLUDE_TITAN_IN_ENSEMBLE and not any(
        name.startswith('titan__') for name in combined_oof_predictions.columns
    ):
        raise RuntimeError('Titan OOFにEmbedding依存モデルがありません。')
    ENSEMBLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    combined_oof_predictions.to_parquet(ENSEMBLE_OUTPUT_DIR / 'oof_predictions.parquet', index=True)
    print('ensemble candidates:', combined_oof_predictions.columns.tolist())
    comparison_rows = []
    for profile, settings in ENSEMBLE_PROFILES.items():
        if len(settings['fold_weights']) != len(folds):
            raise RuntimeError(f'{profile}のfold_weightsを実際のfold数に合わせてください。')
        result = hill_climb_auc(
            combined_oof_predictions, train[TARGET_COL],
            candidate_models=ENSEMBLE_CANDIDATES, folds=folds,
            max_steps=50, weight_grid=np.arange(0.05, 0.55, 0.05),
            min_improvement=1e-6, **settings,
        )
        ensemble_results[profile] = result
        fold_auc = evaluate_fold_auc(
            result.oof_prediction, train[TARGET_COL], folds, years=train[YEAR_COL]
        )
        comparison_rows.append({
            'profile': profile, 'objective_score': result.score,
            'pooled_auc': result.pooled_auc, 'mean_fold_auc': fold_auc['roc_auc'].mean(),
            'latest_fold_auc': fold_auc.iloc[-1]['roc_auc'],
            'blend_mode': result.blend_mode, 'n_models': int(result.weights.gt(0).sum()),
            'fold_weights': str(settings['fold_weights']),
        })
        save_ensemble_outputs(result, ENSEMBLE_OUTPUT_DIR / 'ensembles' / profile)
    ensemble_comparison = pd.DataFrame(comparison_rows).set_index('profile')
    ensemble_comparison.to_csv(ENSEMBLE_OUTPUT_DIR / 'ensemble_profile_comparison.csv')
    display(ensemble_comparison)
    ensemble_result = ensemble_results[SELECTED_ENSEMBLE_PROFILE]
    display(ensemble_result.weights[ensemble_result.weights > 0].sort_values(ascending=False).to_frame())
else:
    print('Ensemble is disabled. Set RUN_ENSEMBLE=True after CV.')

## 6. Cohere/Titanを全train再fitしてprofile別submission作成

全profileで正のweightを持つモデルの和集合だけを一度ずつ再学習します。`cohere__`モデルにはCohere cache、`titan__`モデルにはNotebook 03のTitan cacheを使います。共有E5/T1/T2は一度だけ学習します。出力先は`outputs/cohere_titan_ensemble/`です。

In [ ]:
if RUN_FINAL_SUBMISSION:
    if not ensemble_results or suite is None:
        raise RuntimeError('先にRUN_CV=True、RUN_ENSEMBLE=Trueを実行してください。')
    final_experiments = sorted({
        experiment
        for result in ensemble_results.values()
        for experiment in result.weights[result.weights > 0].index
    })
    selected_sources = {
        split_prediction_namespace(name, namespaces={'cohere', 'titan'})[0]
        for name in final_experiments
    }
    if 'cohere' in selected_sources and (train_embeddings is None or test_embeddings is None):
        raise RuntimeError('選抜されたcohere__モデルにはtrain/test Cohere Embeddingが必要です。')

    titan_train_embeddings = titan_test_embeddings = None
    titan_train_metadata = titan_test_metadata = None
    if 'titan' in selected_sources:
        if TITAN_EMBEDDING_CACHE_DIR is None:
            raise RuntimeError('選抜されたtitan__モデル用にTITAN_EMBEDDING_CACHE_DIRを指定してください。')
        titan_train_embeddings, titan_train_metadata = load_embeddings(
            TITAN_EMBEDDING_CACHE_DIR, 'train', expected_df=train, project_id_col=ID_COL
        )
        titan_test_embeddings, titan_test_metadata = load_embeddings(
            TITAN_EMBEDDING_CACHE_DIR, 'test', expected_df=test, project_id_col=ID_COL
        )

    prediction_dir = ENSEMBLE_OUTPUT_DIR / 'test_predictions'
    prediction_dir.mkdir(parents=True, exist_ok=True)
    test_predictions = {}
    for prediction_name in final_experiments:
        source, experiment = split_prediction_namespace(
            prediction_name, namespaces={'cohere', 'titan'}
        )
        if source == 'cohere':
            source_train_emb, source_test_emb = train_embeddings, test_embeddings
            source_train_meta, source_test_meta = train_embedding_metadata, test_embedding_metadata
        elif source == 'titan':
            source_train_emb, source_test_emb = titan_train_embeddings, titan_test_embeddings
            source_train_meta, source_test_meta = titan_train_metadata, titan_test_metadata
        else:
            source_train_emb = source_test_emb = None
            source_train_meta = source_test_meta = None

        if experiment in suite.results:
            pca_dim = suite.results[experiment].metadata.get('pca_dim')
        elif experiment.startswith('E6_embedding_tabular_xgb_pca'):
            pca_dim = int(experiment.removeprefix('E6_embedding_tabular_xgb_pca'))
        else:
            pca_dim = None
        prediction = fit_full_and_predict_test(
            experiment=experiment, train=train, test=test,
            train_embeddings=source_train_emb, test_embeddings=source_test_emb,
            train_embedding_metadata=source_train_meta,
            test_embedding_metadata=source_test_meta,
            numeric_cols=NUMERIC_COLS, categorical_cols=CATEGORICAL_COLS,
            config=CONFIG, pca_dim=pca_dim, target_col=TARGET_COL, project_id_col=ID_COL,
        )
        test_predictions[prediction_name] = prediction
        np.save(prediction_dir / f'{prediction_name}.npy', prediction.astype(np.float32))
        print(prediction_name, prediction.shape, prediction.min(), prediction.max())

    submissions, submission_manifest = make_profile_submissions(
        test, test_predictions, ensemble_results,
        id_col=ID_COL, prediction_col=TARGET_COL,
        output_dir=ENSEMBLE_OUTPUT_DIR / 'submissions',
        canonical_profile=SELECTED_ENSEMBLE_PROFILE,
        canonical_output_path=ENSEMBLE_OUTPUT_DIR / 'submission.csv',
    )
    canonical = pd.read_csv(ENSEMBLE_OUTPUT_DIR / 'submission.csv')
    assert canonical.columns.tolist() == [ID_COL, TARGET_COL]
    assert canonical[ID_COL].astype(str).tolist() == test[ID_COL].astype(str).tolist()
    assert len(canonical) == len(test)
    display(submission_manifest)
    display(canonical.head())
    print('canonical:', ENSEMBLE_OUTPUT_DIR / 'submission.csv')
else:
    print('Final submission is disabled. Set RUN_FINAL_SUBMISSION=True after ensemble review.')